In [1]:
import pandas as pd
import os

filepath = "../DATA-HTML-STOCK/NEPSECompany/NEPSECompanyExtractor.csv"
symbols = pd.read_csv(filepath)['Symbol'].tolist()

stock_ignored = ["CORBLP", "GFCLPO", "HBLPO", "LFCPO", "SBIPO", "GABLPO", "AMFIPO", "NMLBS", "ILFCPO"]
stock_no_news = ["HATHPO", "BUDBLP", "CFCLPO", "CORBLP", "EBLPO", "HAMROP", "HGIPO", "JFLPO", "JBNLPO", "KADBLP", "KNBLPO", "CEFLPO", "LBLPO", "MFILPO", "MIDBLP", "NBBLPO", "NABBPO", "NBBPO", "PFLPO", "PRINPO", "PURBLP", "SBBLJP", "SIFCPO", "SILPO", "SMFDBP", "SYFLPO", "TNBLPO", "TDBLPO", "UFLPO", "WDBLPO", "WMBFPO", "HLBSLP"]

def detailed_prediction(score):
    if score >= 0.60:   return "Strongly Positive"
    elif score >= 0.20: return "Positive"
    elif score >= 0.05: return "Slightly Positive"
    elif score > -0.05: return "Neutral"
    elif score > -0.20: return "Slightly Negative"
    elif score > -0.60: return "Negative"
    else:               return "Strongly Negative"

def apply_3day_decay(df):
    df = df.sort_values("Date_Stock", ascending=True).reset_index(drop=True)
    t1, t2, t3 = 0.0, 0.0, 0.0
    decayed = []
    for i, row in df.iterrows():
        score = row["Sentiment_Score"]
        if i == 0:
            t1 = score
        else:
            t3 = t2 / 2
            t2 = t1 / 2
            t1 = score
        decayed.append(t1 + t2 + t3)
    df["Decayed_Sentiment"] = decayed
    df["Decayed_Prediction"] = df["Decayed_Sentiment"].apply(detailed_prediction)
    return df.sort_values("Date_Stock", ascending=False).reset_index(drop=True)

for symbol in symbols:
    if symbol in stock_ignored:
        print(f"{symbol} is in ignore list!")
        continue

    semifinal_path = f"../DATA-HTML-STOCK/SemiFinalDataset/{symbol}.csv"
    if not os.path.exists(semifinal_path):
        print(f"{symbol} no semifinal data found, skipping")
        continue

    semifinal_df = pd.read_csv(semifinal_path)
    savepath = f"../DATA-HTML-STOCK/FinalDataSet/{symbol}.csv"

    if os.path.exists(savepath):
        old_df = pd.read_csv(savepath)
        saved_date = old_df["Date_Stock"].iloc[0]

        new_rows = semifinal_df[semifinal_df["Date_Stock"] > saved_date]
        if new_rows.empty:
            print(f"{symbol} no new data for decay")
            continue

        new_rows = new_rows.sort_values("Date_Stock", ascending=True).reset_index(drop=True)
        old_sorted = old_df.sort_values("Date_Stock", ascending=True)
        t1 = old_sorted["Decayed_Sentiment"].iloc[-1]
        t2 = old_sorted["Decayed_Sentiment"].iloc[-2] if len(old_sorted) > 1 else 0.0
        t3 = 0.0
        decayed = []
        for _, row in new_rows.iterrows():
            t3 = t2 / 2
            t2 = t1 / 2
            t1 = row["Sentiment_Score"]
            decayed.append(t1 + t2 + t3)
        new_rows["Decayed_Sentiment"] = decayed
        new_rows["Decayed_Prediction"] = new_rows["Decayed_Sentiment"].apply(detailed_prediction)

        final_df = pd.concat([old_df, new_rows], ignore_index=True)
        final_df = final_df.sort_values("Date_Stock", ascending=False).reset_index(drop=True)
    else:
        if symbol in stock_no_news:
            print(f"{symbol} is in stock no news list!")
            semifinal_df["Decayed_Sentiment"] = 0.0
            semifinal_df["Decayed_Prediction"] = "Neutral"
            final_df = semifinal_df
        else:
            final_df = apply_3day_decay(semifinal_df)

    print(final_df)
    final_df.to_csv(savepath, index=False)

      Date_Stock  Close Date_News  Sentiment_Score Prediction  \
0     2026-03-03  305.0       NaN              0.0    Neutral   
1     2026-03-01  302.6       NaN              0.0    Neutral   
2     2026-02-26  298.0       NaN              0.0    Neutral   
3     2026-02-25  294.6       NaN              0.0    Neutral   
4     2026-02-24  293.0       NaN              0.0    Neutral   
...          ...    ...       ...              ...        ...   
3523  2010-09-12  118.0       NaN              0.0    Neutral   
3524  2010-09-09  122.0       NaN              0.0    Neutral   
3525  2010-09-08  125.0       NaN              0.0    Neutral   
3526  2010-09-07  138.0       NaN              0.0    Neutral   
3527  2010-09-02  255.0       NaN              0.0    Neutral   

      Decayed_Sentiment Decayed_Prediction  
0                   0.0            Neutral  
1                   0.0            Neutral  
2                   0.0            Neutral  
3                   0.0            Neut